# RG3 미성형 원상태/다른상태 D 분기


In [ ]:
from pathlib import Path
import os,nbformat,json
import numpy as np,pandas as pd
from sklearn.metrics import confusion_matrix
ROOT=Path.cwd();WORK=ROOT/'rg3/internal';tables=WORK/'reports/tables'
s=''.join(c.source for c in nbformat.read(WORK/'notebooks/208_rg3_shortshot_D_alarm_correction.ipynb',4).cells if c.cell_type=='code')
ns={'__name__':'__main__'};exec(s[:s.index('records=[];audits=[]')],ns)
lab=ns['lab'];y=ns['y'];g=ns['g'];use=ns['use']
support=np.zeros(len(lab),bool)
for block in pd.unique(g):
 q=np.flatnonzero(g==block);ref=np.flatnonzero((g!=block)&use)
 support[q]=ns['temp_support'](ref,ns['context'][q])[0]
base=pd.read_csv(tables/'rg3_204_shortshot_reduction_predictions.csv',parse_dates=['TimeStamp'])
base=base[base.candidate.eq('pruned15')].set_index('TimeStamp').loc[lab.TimeStamp]
abc=base[['A','B','C']].any(axis=1).to_numpy()
new=pd.read_csv(tables/'rg3_209_conditioned_D_predictions.csv',parse_dates=['TimeStamp'])
u=ns['u'];diag=ns['diag'];ref=np.flatnonzero(use)
usupport=ns['temp_support'](ref,ns['ctxU'])[0]
uabc=diag[['A','B','C']].any(axis=1).to_numpy();uold=diag.D.to_numpy(bool)
records=[];metrics=[];um=[]
gas=pd.read_csv(tables/'rg3_194_fresh12_unlabeled_predictions.csv',parse_dates=['TimeStamp']).set_index('TimeStamp').loc[u.TimeStamp].gas_candidate.eq(1).to_numpy()
for quant in [.99,.995,.999]:
 values=new[new.policy.eq(quant)].set_index('TimeStamp').loc[lab.TimeStamp]
 nd=values.D.to_numpy(bool);od=base.D.to_numpy(bool)
 un=pd.read_csv(tables/('rg3_209_conditioned_unlabeled_'+str(quant)+'.csv'),parse_dates=['TimeStamp']).set_index('TimeStamp').loc[u.TimeStamp]
 for rule in ['fallback','union']:
  dd=(od&support)|(nd&~support) if rule=='fallback' else (od&support)|nd
  pr=abc|dd;policy=rule+'_'+str(quant)
  mask=use
  tn,fp,fn,tp=confusion_matrix(y[mask],pr[mask],labels=[0,1]).ravel()
  metrics.append(dict(policy=policy,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=float(2*tp/(2*tp+fp+fn))))
  for i,ts in enumerate(lab.TimeStamp):records.append(dict(policy=policy,TimeStamp=ts,type=lab.type.iloc[i],actual=int(y[i]),prediction=int(pr[i]),D=int(dd[i]),old_state_supported=int(support[i])))
  und=un.D.to_numpy(bool)
  udd=(uold&usupport)|(und&~usupport) if rule=='fallback' else (uold&usupport)|und
  upr=uabc|udd;combined=upr|gas
  um.append(dict(policy=policy,N=len(u),shortshot_candidates=int(upr.sum()),shortshot_percent=float(100*upr.mean()),gas_candidates=int(gas.sum()),overlap=int((upr&gas).sum()),combined_candidates=int(combined.sum()),combined_percent=float(100*combined.mean())))
  pd.DataFrame({'TimeStamp':u.TimeStamp,'shortshot_prediction':upr.astype(int),'gas_prediction':gas.astype(int),'combined_prediction':combined.astype(int),'D':udd.astype(int),'old_state_supported':usupport.astype(int)}).to_csv(tables/('rg3_210_'+policy+'_unlabeled.csv'),index=False)
pp=pd.DataFrame(records);mm=pd.DataFrame(metrics);uu=pd.DataFrame(um)
pp.to_csv(tables/'rg3_210_D_routing_predictions.csv',index=False)
mm.to_csv(tables/'rg3_210_D_routing_metrics.csv',index=False)
uu.to_csv(tables/'rg3_210_D_routing_unlabeled_metrics.csv',index=False)
print(mm.to_string(index=False));print(uu.to_string(index=False))
text='# RG3 미성형 상태별 D 분기 보완\n\n원래정상참조온도상태안에서는기존D,밖에서는과거비라벨상태기준선으로재정규화한D를사용한다(fallback). union은조건지원기존D와새D둘다허용하는개발비교. 분류기학습에는실제라벨만사용하고,과거비라벨은중앙값/MAD통계에만활용한다. 알수없는샷을정상으로바꿔집계하지않고전체574샷을채점했다.\n\n'+mm.to_string(index=False)+'\n\n'+uu.to_string(index=False)
text+='\n\n입력라벨샷분리및정상참조/임계값학습은208/209에정의된다. 기존ABC세부분은고정. 동일자료를반복검토한개발선택이므로새독립검증아님. 과거비라벨참조에다수불량이섞이면기준선이불량을흡수할위험이남음.6590샷은기존조건유사필터통과부분집합,비라벨예측진위모름.예측은학습후모델의역사자료소급적용이며온라인미래검증아님.원배포모델및Git미교체.\n'
(WORK/'reports/rg3_210_D_routing_review.md').write_text(text,encoding='utf-8')

